# Preprocess The Raw Dataset for Tree Model

### 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Get the path to 'irrigation-prediction-system' (the project root)
root_path: Path = Path.cwd().parent.parent.parent
sys.path.append(str(root_path))

### 2. Import Required Libraries and Custom Configs

In [ ]:
import joblib
import numpy as np
import pandas as pd
from prefect.settings import PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW, temporary_settings
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from src.configs.data_cfg import FALSE_VALUES, NA_VALUES, OPTIMIZED_DTYPES, TRUE_VALUES
from src.configs.settings import Settings, get_settings
from src.pipeline.ingestion import load_dataset

# Setup configs
settings: Settings = get_settings()

# Setup data paths
raw_data_path: Path = settings.RAW_DATA_DIR / settings.RAW_DATA_FILENAME
train_filepath: Path = settings.EXPERIMENTS_DATA_DIR / settings.TREE_TRAIN_FILENAME
test_filepath: Path = settings.EXPERIMENTS_DATA_DIR / settings.TREE_TEST_FILENAME
preprocess_filepath: Path = settings.EXP_ARTIFACT_DIR / settings.PREPROCESS_PIPELINE_FILENAME
target_encoder_filepath: Path = settings.EXP_ARTIFACT_DIR / settings.TARGET_ENCODER_FILENAME

### 3. Load the Dataset

In [ ]:
try:
    with temporary_settings(updates={PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW: "ignore"}):
        df: pd.DataFrame = load_dataset(
            raw_data_path,
            sep=",",
            header=0,
            index_col="id",
            true_values=TRUE_VALUES,
            false_values=FALSE_VALUES,
            dtype=OPTIMIZED_DTYPES,
            na_values=NA_VALUES,
            keep_default_na=True,
            on_bad_lines="warn",
            float_precision="round_trip",
            skipinitialspace=True,
            encoding="utf-8",
            encoding_errors="replace",
            memory_map=True,
            low_memory=False,
        )
except Exception as e:
    sys.stderr.write(f"Ingestion failed: {e}\n")

### 4. Split the Columns into Groups

In [ ]:
# Target
target_col: str = "Irrigation_Need"

# Feature groups
bool_cols: tuple[str, ...] = ("Mulching_Used",)
ordinal_cols: tuple[str, ...] = ("Crop_Growth_Stage",)
nominal_cols: tuple[str, ...] = (
    "Soil_Type",
    "Crop_Type",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
)

# Ordinal category orders
crop_growth_stage_categories: tuple[str, ...] = ("Sowing", "Vegetative", "Flowering", "Harvest")
irrigation_need_categories: tuple[str, ...] = ("Low", "Medium", "High")

# Numeric and categorica columns
numeric_cols: list[str] = df.select_dtypes(["Float32"]).columns.tolist()
categorical_cols: list[str] = df.select_dtypes(["category", "boolean"]).columns.tolist()
categorical_cols.remove(target_col)

### 5. Split the Dataset into Train-Test

In [ ]:
# Separate features (X) and target (y)
X: pd.DataFrame = df.drop(columns=[target_col])
y: pd.Series = df[target_col]

# Encode the target before split
encoder: OrdinalEncoder = OrdinalEncoder(categories=[list(irrigation_need_categories)])
y_encoded = encoder.fit_transform(y.to_numpy().reshape(-1, 1))
y = pd.Series(y_encoded.flatten(), index=y.index, name=target_col, dtype=np.int8)

# Securely serialize the fitted pipeline
joblib.dump(encoder, target_encoder_filepath)
print(f"Preprocessor saved to: {target_encoder_filepath}")

# Stratified train-test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 7. Create Function Transformers for Custom Encoding

In [ ]:
def _make_ordinal_transformer() -> OrdinalEncoder:
    """
    Create ordinal encoder with predefined category ordering.

    Returns
    -------
    OrdinalEncoder
        Encoder with explicit category mapping.
    """
    mapping: dict[str, list[str]] = {"Crop_Growth_Stage": list(crop_growth_stage_categories)}
    categories: list[list[str]] = [mapping[col] for col in ordinal_cols]
    return OrdinalEncoder(
        categories=categories, handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int8
    )


def _make_nominal_transformer() -> OrdinalEncoder:
    """
    Create nominal encoder.

    Returns
    -------
    OrdinalEncoder
        Encoder with explicit category mapping.
    """
    return OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1, dtype=np.int8
    )

### 8. Build the Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("bool", OrdinalEncoder(dtype=np.int8), list(bool_cols)),
        ("ordinal", _make_ordinal_transformer(), list(ordinal_cols)),
        ("nominal", _make_nominal_transformer(), list(nominal_cols)),
        ("numeric", "passthrough", numeric_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

pipeline: Pipeline = Pipeline(steps=[("preprocessor", preprocessor)], verbose=True)
pipeline.set_output(transform="pandas")

### 9. Encode the Datasets and Save Locally

In [ ]:
pipeline.fit(X_train)

# Securely serialize the fitted pipeline
joblib.dump(pipeline, preprocess_filepath)
print(f"Preprocessor saved to: {preprocess_filepath}")

# Transform the data
X_train_enc: pd.DataFrame = pipeline.transform(X_train)
X_test_enc: pd.DataFrame = pipeline.transform(X_test)
X_train_enc.info()

### 10. Save the datasets as parquet

In [ ]:
pd.concat([X_train_enc, y_train], axis=1).to_parquet(train_filepath)
pd.concat([X_test_enc, y_test], axis=1).to_parquet(test_filepath)
print(f"Data saved at {settings.EXPERIMENTS_DATA_DIR}")

In [ ]:
# Stack the train and test sets vertically
df_full = pd.concat(
    [pd.concat([X_train_enc, y_train], axis=1), pd.concat([X_test_enc, y_test], axis=1)], axis=0
)

df_full.to_parquet(settings.EXPERIMENTS_DATA_DIR / settings.TREE_FILENAME)
print(f"Full dataset saved at {settings.EXPERIMENTS_DATA_DIR}")